# Assignment 1: exploratory data analysis and baselines

**Group 43, CENG313** · Mehmet Emircan Küllücek, Aleksei Khlopkov

Competition: [PhysioNet – Digitization of ECG Images](https://www.kaggle.com/competitions/physionet-ecg-image-digitization). It is closed, so we score locally on training records whose true signals we have.

**Task.** Given a picture of a printed 12-lead ECG, recover the 12 voltage curves (mV over time) that were printed on it.

This notebook
1. looks at the records, the ground-truth signals and the nine kinds of images,
2. tests how the official metric reacts to typical errors,
3. checks the page layout that our digitizer depends on,
4. scores two baselines with the official metric: **all zeros** and a **classical computer-vision digitizer** (no learning).

All reusable code is in `src/` (data loading, the official metric, the CV baseline, the page template). The notebook only calls it.

**Setup.** In Colab, first add a secret named `GITHUB_TOKEN` (key icon in the left sidebar) with a GitHub token that can read this private repo. The cell below clones the repo and downloads our dataset subset from Google Drive (about 2.5 GB, first run only). Running locally, start Jupyter from the repo root.

In [ ]:
# In Colab: clone the private repo with the GITHUB_TOKEN secret, then fetch the data.
# Locally: start Jupyter from the repo root and this only fetches the data if missing.
import os, sys

BRANCH = "main"  # set to a feature branch to run that branch's code

if "google.colab" in sys.modules and not os.path.isdir("src"):
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    !git clone -q -b {BRANCH} https://{token}@github.com/mmilesmorales/GROUP43_CENG313.git
    del token
    %cd GROUP43_CENG313

sys.path.insert(0, "src")
from data import download_dataset
download_dataset()  # ~2.5 GB on the first run, skipped when data/ exists
print("data ready:", len(os.listdir("data/train")), "record folders")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from data import load_val_ids, load_record_gt, extract_lead_signal
from metric import record_linear_snr, competition_score_from_linear
from utils import einthoven_residual
from baseline_cv import run_validation
import layout

DATA = "data"
LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
IMAGE_TYPES = ["0001", "0003", "0004", "0005", "0006", "0009", "0010", "0011", "0012"]
REC = 1006427285  # example record shown in the plots

train = pd.read_csv(f"{DATA}/train.csv")
fs_of = dict(zip(train.id, train.fs))
val_ids = load_val_ids()
eval_ids = val_ids[:30]  # fixed evaluation subset for all scores in this notebook
gt_ids = [r for r in val_ids if os.path.exists(f"{DATA}/train/{r}/{r}.csv")]
print(f"train.csv: {len(train)} records | validation split: {len(val_ids)} | "
      f"with ground truth in our subset: {len(gt_ids)} | evaluation subset: {len(eval_ids)}")

## 1. Dataset structure
- `train.csv`: 977 records `id, fs, sig_len` (fs 250–1025 Hz; sig_len = 10 s × fs)
- `train/<id>/<id>.csv`: ground truth (12 lead columns)
- `train/<id>/<id>-<variant>.png`: **9 imaging variants** per record
- submission rows per lead: `floor(fs*2.5)` short leads, `floor(fs*10)` lead II

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
tr.fs.value_counts().sort_index().plot.bar(ax=ax[0], title="fs distribution (train)")
tr.sig_len.value_counts().sort_index().plot.bar(ax=ax[1], title="sig_len distribution")
for a in ax: a.set_yscale("log")
plt.tight_layout(); plt.show()
print("sig_len == 10*fs always:", bool((tr.sig_len == 10*tr.fs).all()))

## 2. Ground truth — one record, 12 leads
Each lead is its own 2.5 s series (lead II: 10 s). Lead layout note: the CSV stores the
12 leads in 2.5-second serialization blocks; `extract_lead_signal` handles that.
Also below: paper layout verification (rows [I,aVR,V1,V4]/[II,aVL,V2,V5]/[III,aVF,V3,V6]
+ full-width rhythm strip) — the constants the CellularVision baseline relies on.

In [ ]:
rec = 1006427285
gt = load_record_gt(DATA, rec)
fig, axes = plt.subplots(1, 12, figsize=(18, 2))
for i, c in enumerate(gt.columns):
    s = extract_lead_signal(gt, c)
    axes[i].plot(s[:2562], lw=0.4); axes[i].set_title(c, fontsize=8)
plt.suptitle(f"Ground truth, record {rec} (mV)"); plt.tight_layout(); plt.show()

## 3. The 9 imaging variants — same signal, deteriorations applied

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 12))
for ax, v in zip(axes.ravel(), VARIANTS):
    p = f"{DATA}/train/{rec}/{rec}-{v}.png"
    ax.imshow(Image.open(p)); ax.set_title(f"{v}  {os.path.getsize(p)/1e6:.1f} MB", fontsize=9); ax.axis("off")
plt.suptitle(f"Record {rec}: 9 imaging variants", fontsize=13); plt.tight_layout(); plt.show()

## 4. Paper layout & calibration (verified)

In [ ]:
im = np.asarray(Image.open(f"{DATA}/train/{rec}/{rec}-0001.png").convert("RGB"))
r, g, b = im[...,0].astype(int), im[...,1].astype(int), im[...,2].astype(int)
reddish = (r > g + 20) & (r > b + 20)
black = (im.mean(2) < 110) & ~reddish
def centroid(y0,y1,x0,x1):
    ys=[]
    for x in range(x0, x1):
        c = np.flatnonzero(black[y0:y1, x])
        ys.append(c.mean()+y0 if len(c) else np.nan)
    return np.asarray(ys)
xg, cellw, N = 118, 492, 2562
rows_y = [(628,824),(890,1146),(1195,1425),(1455,1664)]
gt_norm = {L: (lambda s: (s-s.mean())/(s.std()+1e-9))(extract_lead_signal(gt, L)) for L in gt.columns}
names = list(gt.columns)
for bi,(y0,y1) in enumerate(rows_y):
    row=[]
    for ci in range(4):
        ys = centroid(y0,y1,xg+ci*cellw, xg+(ci+1)*cellw)
        fin = np.flatnonzero(np.isfinite(ys))
        s = np.interp(np.linspace(0,1,N), fin/len(ys), ys[fin]); s=(s-s.mean())/(s.std()+1e-9)
        ccs = {L: abs(np.corrcoef(s, gt_norm[L][:N])[0,1]) for L in names}
        row.append(f"{max(ccs, key=ccs.get)}({max(ccs.values()):.2f})")
    print(f"row {chr(65+bi)}:", row)

## 5. Baseline 1 — all zeros (~0 dB anchor)

In [ ]:
val_ids = load_val_ids(30)      # fixed 30-record subset (record-level 90/10 split, seed 0)
have = [r for r in val_ids if os.path.exists(f"{DATA}/train/{r}/{r}.csv")]
ratios = []
for rec in have:
    g = load_record_gt(DATA, rec)
    ratios.append(record_linear_snr(g, {L: np.zeros(len(g[L].dropna())) for L in g.columns}, fs_of[str(rec)]))
print(f"records scored: {len(have)}")
print(f"BASELINE 1 (all zeros): {competition_score_from_linear(ratios):.3f} dB")

## 6. Baseline 2 — classical CV digitization (no ML)
`src/baseline_cv.py`: grid autocorrelation → calibration (px/s, px/mV) → fixed-layout
cell extraction → per-column trace tracking → resample → official metric. Evaluated on
the same 30-record subset across **all 9 variants** (270 evaluations).

In [ ]:
have_pngs = [r for r in val_ids if all(os.path.exists(f"{DATA}/train/{r}/{r}-{v}.png") for v in VARIANTS)]
print("records with full image set:", len(have_pngs))
df_cv = run_validation(have_pngs, out_csv="cv_scores.csv", verbose=False)
print(df_cv.groupby('variant').db.agg(['mean','min','max']).round(2).to_string())
print(f"\nBASELINE 2 (classical CV): {competition_score_from_linear(df_cv.ratio):.3f} dB")

## 7. Results summary

In [ ]:
summary = {
    "all-zeros": competition_score_from_linear(ratios),
    "classical CV": competition_score_from_linear(df_cv.ratio),
    "community pretrained (same cohort, reference)": 17.989,
}
print(f"{'baseline':46s} {'score (dB)':>10s}")
for k, v in summary.items(): print(f"{k:46s} {v:10.3f}")
print("\nreference only — competition winner 23.33 dB on the PRIVATE test set (different data)")

**Takeaways**
- Pure-geometry CV is *correct on clean scans* (up to +15 dB/record) but collapses on
real-world variants (168/270 records below the zeros floor) — quantifying exactly why
successful approaches begin with learned rectification.
- The all-zeros anchor sits at ≈ 0 dB: the metric's vertical-offset removal gives small
positive boosts (≤ +3.9 dB/record).
- Everything is scored with the official metric on a fixed record-level validation split.